In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

In [2]:
problem_name = os.path.basename(os.path.abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 00:36:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
raw_df = spark.read\
.option("header", "true")\
.format("csv")\
.load("files/raw_data.txt")

In [5]:
raw_df.show()

+-----+---+----------+------+----------+
| Name| ID|       DOB|Gender|Occupation|
+-----+---+----------+------+----------+
| John|  1|1990-05-25|  Male|  Engineer|
|Alice|  2|15-08-1985|Female|    Doctor|
|  Bob|  3|2001-11-10|  Male|   Teacher|
| Mary|  4|12-07-1978|Female|  Engineer|
| John|  1|1990-05-25|  Male|  Engineer|
|Alice|  2|15-08-1985|Female|    Doctor|
|  Bob|  3|2001-11-10|  Male|   Teacher|
+-----+---+----------+------+----------+



In [6]:
raw_df\
.withColumn("row_num", row_number().over(Window.partitionBy(raw_df.columns).orderBy("ID")))\
.filter(col("row_num") == 1)\
.select(raw_df.columns)\
.show()

+-----+---+----------+------+----------+
| Name| ID|       DOB|Gender|Occupation|
+-----+---+----------+------+----------+
|Alice|  2|15-08-1985|Female|    Doctor|
|  Bob|  3|2001-11-10|  Male|   Teacher|
| John|  1|1990-05-25|  Male|  Engineer|
| Mary|  4|12-07-1978|Female|  Engineer|
+-----+---+----------+------+----------+



In [7]:
spark.stop()